# Graph-Grounded Issue Localization: Q1-Ready Experimental Pipeline

**Primary research question:** when and why does repository structure improve software issue localization, and does the same structural information help constrained LLM agents?

This upgraded notebook preserves the original BM25 → BM25+signals → GraphRank ladder, expands evaluation to the full available task set, adds held-out graph-weight selection, robustness/sensitivity analyses, graph-component diagnostics, task-level analysis, and controlled A/S/D agent evaluation when the model environment is available.

**No result is simulated or back-filled. If an experiment cannot run, it is explicitly marked unavailable.**


In [1]:
from pathlib import Path

W = Path("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse")

print("Exists:", W.exists())

for f in W.glob("*.whl"):
    print(f.name)

Exists: True
safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
mistral_common-1.11.7-py3-none-any.whl
partial_json_parser-0.2.1.1.post7-py3-none-any.whl
compressed_tensors-0.15.0.1-py3-none-any.whl
setproctitle-1.3.7-cp312-cp312-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl
adk_submission-0.2.12-py3-none-any.whl
model_hosting_container_standards-0.1.16-py3-none-any.whl
interegular-0.3.3-py37-none-any.whl
outlines_core-0.2.11-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
msgspec-0.21.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
pycountry-26.2.16-py3-none-any.whl
adk_eval_core-0.1.0-py3-none-any.whl
quack_kernels-0.6.4-py3-none-any.whl
pydantic_extra_types-2.11.2-py3-none-any.whl
transformers-5.13.1-py3-none-any.whl
apache_tvm_ffi-0.1.13.post3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl
google_adk-1.36.1-py3-none-any.whl
google_genai-2.11.0-py3-none-any.whl
gguf-0.19.0-py3-none

In [2]:
!pip install --no-index --no-deps \
  /kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse/vllm-0.19.1-cp38-abi3-manylinux_2_31_x86_64.whl

Processing /kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse/vllm-0.19.1-cp38-abi3-manylinux_2_31_x86_64.whl


In [3]:
# Offline dependency repair for Phase 2. Never use the internet in the experiment environment.
# The supplied wheelhouse contains msgspec. Install only if a compatible wheel exists.
import sys, subprocess
#WHEELHOUSE = Path("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse")
WHEELHOUSE = Path("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse")
print("Python:", sys.version)
print("Wheelhouse exists:", WHEELHOUSE.exists())
if WHEELHOUSE.exists():
    wheels = sorted(WHEELHOUSE.glob("msgspec-*.whl"))
    print("msgspec wheels:", [w.name for w in wheels])
    if wheels:
        r = subprocess.run([sys.executable,"-m","pip","install","--no-index","--no-deps",str(wheels[0])],capture_output=True,text=True)
        print(r.stdout[-1200:]); print(r.stderr[-1200:])
        try:
            import msgspec; print("msgspec:", msgspec.__version__)
        except Exception as e: print("msgspec unavailable:",repr(e))
else:
    print("Attach the competition wheelhouse dataset before Phase 2.")


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Wheelhouse exists: True
msgspec wheels: ['msgspec-0.21.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl']

ERROR: msgspec-0.21.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl is not a supported wheel on this platform.

msgspec unavailable: ModuleNotFoundError("No module named 'msgspec'")


## 0. Setup and configuration
Imports, helper to locate inputs, and the single `CFG` dictionary that controls the whole run.

In [4]:
import os, sys, json, time, platform, subprocess, zipfile, hashlib, inspect, copy, random
from pathlib import Path
import numpy as np

INPUT_ROOT = Path(os.environ.get("GGIL_INPUT_ROOT", "/kaggle/input"))
def walk_find(pred, root=INPUT_ROOT, maxdepth=7):
    hits=[]
    if not root.exists(): return hits
    base=len(root.parts)
    for dp,dns,fns in os.walk(root):
        if len(Path(dp).parts)-base >= maxdepth: dns[:]=[]
        for n in dns+fns:
            p=Path(dp)/n
            if pred(p): hits.append(p)
    return sorted(hits)

CFG=dict(
    TOOLKIT="/kaggle/input/datasets/manishverma649/ggil-toolkit/ggil_pkg",
    TASKS="/kaggle/input/competitions/gemma-4-developer-agent/tasks.jsonl",
    SNAPSHOTS="/kaggle/input/competitions/gemma-4-developer-agent/snapshots",
    ID_FILE=os.environ.get("GGIL_IDS"),
    LIMIT=int(os.environ.get("GGIL_LIMIT",129)),
    FIELD_MAP={}, ALLOW_UNVERIFIED=False,
    MODEL_PATH=os.environ.get("GGIL_MODEL"), VARIANTS=["A","S","D"],
    MAX_STEPS=12, MAX_MODEL_LEN=16384, MAX_NEW_TOKENS=512, TEMPERATURE=0.0,
    GPU_MEM_UTIL=0.80, TENSOR_PARALLEL_SIZE=int(os.environ.get("GGIL_TP",4)),
    MAX_MINUTES=int(os.environ.get("GGIL_MAX_MINUTES",480)),
    BOOTSTRAP=5000, RANDOM_SEED=2026, PROPAGATION_WEIGHT=0.30,
    ALPHA_GRID=[0.0,0.1,0.2,0.3,0.4,0.5,0.7,0.9], MAX_HOPS=4,
    OUT=Path(os.environ.get("GGIL_OUT","/kaggle/working/ggil_results_q1")), PACK=False)
CFG["OUT"].mkdir(parents=True,exist_ok=True)
random.seed(CFG["RANDOM_SEED"])
print({k:(str(v) if isinstance(v,Path) else v) for k,v in CFG.items() if k!="FIELD_MAP"})


{'TOOLKIT': '/kaggle/input/datasets/manishverma649/ggil-toolkit/ggil_pkg', 'TASKS': '/kaggle/input/competitions/gemma-4-developer-agent/tasks.jsonl', 'SNAPSHOTS': '/kaggle/input/competitions/gemma-4-developer-agent/snapshots', 'ID_FILE': None, 'LIMIT': 129, 'ALLOW_UNVERIFIED': False, 'MODEL_PATH': None, 'VARIANTS': ['A', 'S', 'D'], 'MAX_STEPS': 12, 'MAX_MODEL_LEN': 16384, 'MAX_NEW_TOKENS': 512, 'TEMPERATURE': 0.0, 'GPU_MEM_UTIL': 0.8, 'TENSOR_PARALLEL_SIZE': 4, 'MAX_MINUTES': 480, 'BOOTSTRAP': 5000, 'RANDOM_SEED': 2026, 'PROPAGATION_WEIGHT': 0.3, 'ALPHA_GRID': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 0.9], 'MAX_HOPS': 4, 'OUT': '/kaggle/working/ggil_results_q1', 'PACK': False}


## Q1 study design

**RQ1:** Do issue-specific traceback/identifier signals improve lexical retrieval?

**RQ2:** Does graph propagation add value beyond those signals?

**RQ3:** How sensitive are results to propagation weight, graph relation type, graph noise, and snapshot quality?

**RQ4:** When does graph information help across task and repository characteristics?

**RQ5:** Does graph access improve a small LLM agent under the same context and tool budget?

**RQ6:** Do static graph hints and interactive graph tools differ in accuracy and search efficiency?

The primary run uses all resolvable tasks available in the task file. A clean-snapshot sensitivity analysis excludes tasks whose patch references files absent from the pre-fix snapshot.


## 1. Load the toolkit and self-test it

In [5]:
tk = CFG["TOOLKIT"]
if not tk:
    c = walk_find(lambda p: p.name == "ggil_toolkit.zip") or walk_find(lambda p: p.is_dir() and (p / "ggil" / "__init__.py").exists())
    assert c, "Toolkit not found: add a dataset containing ggil_toolkit.zip, or set CFG['TOOLKIT']."
    tk = str(c[0])
tk = Path(tk)

if tk.suffix == ".zip":
    dst = Path("/tmp/ggil_src"); dst.mkdir(exist_ok=True)
    zipfile.ZipFile(tk).extractall(dst)
    pkg_root = next(dst.rglob("ggil/__init__.py")).parent.parent
else:
    pkg_root = tk if (tk / "ggil").exists() else tk.parent
sys.path.insert(0, str(pkg_root))
import ggil
from ggil import (build_graph, GraphTools, gold_from_patch, parse_patch, score_instance,
                  aggregate, paired_bootstrap)
from ggil.agent import TaskCtx, run_variant
print("toolkit:", pkg_root)

try:
    import pytest  # noqa
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-x", str(pkg_root / "tests")], capture_output=True, text=True, cwd=pkg_root)
    print("self-tests:", r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-300:])
    assert r.returncode == 0, r.stdout[-2000:]
except ImportError:
    print("pytest not installed - skipping self-tests")

toolkit: /kaggle/input/datasets/manishverma649/ggil-toolkit/ggil_pkg
self-tests: 16 passed, 1 warning in 0.19s


## 2. Load tasks
The loader auto-detects common field names (`instance_id`/`id`, `problem_statement`/`issue`/..., `patch`/`gold_patch`/...). It prints what it found so you can verify the mapping.

In [6]:
# --- Task loader (handles SWE-bench-style repo + base_commit layouts); overrides the toolkit's older loader. ---
_HOT = Path('/tmp/ggil_hot'); _HOT.mkdir(exist_ok=True)
(_HOT / 'ggil_data_v2.py').write_text(r'''"""data.py - tolerant task loader (jsonl / json) with snapshot resolution.

Supports SWE-bench-style rows (instance_id, repo, base_commit, problem_statement, patch) and
several snapshot layouts under `snapshots_root`:

  per-instance   <snap>/<instance_id>/            (dir)  or  <snap>/<instance_id>.tar.gz|.tgz|.tar|.zip
                 <snap>/<instance_id>/repo/
                 <snap>/<repo_key>/<instance_id>/
  git@commit     <snap>/<repo_key>/  is a git clone (or bare mirror) -> `git archive <base_commit>`
  per-commit     <snap>/<repo_key>/<base_commit>/  (full or 7/12-char prefix)
  per-repo       <snap>/<repo_key>/  plain checkout, commit NOT verifiable  (flagged!)
  fuzzy          any dir/archive (depth<=3) whose normalized name equals/contains the instance id

`repo_key` candidates come from the repo id ("owner/name" -> owner__name, owner_name, name, ...)
and from the instance-id prefix ("fastapi_11194" -> "fastapi").
Each task gets `snapshot_kind` so you can see how it was resolved; "per-repo" means the commit
could not be verified and gold line numbers may be off.
"""
from __future__ import annotations

import json, os, re, shutil, subprocess, tarfile, zipfile
from collections import Counter
from pathlib import Path
from typing import Dict, List, Optional, Tuple

FIELD_CANDIDATES = {
    "instance_id": ["instance_id", "id", "task_id"],
    "issue": ["problem_statement", "issue", "issue_text", "issue_body", "text", "description", "prompt"],
    "patch": ["patch", "gold_patch", "fix_patch", "solution_patch", "golden_patch"],
    "repo": ["repo_path", "repo_dir", "snapshot", "snapshot_path", "repo_root", "path", "workdir"],  # a PATH
    "repo_id": ["repo", "repository", "repo_name", "repo_slug"],                                     # "owner/name"
    "commit": ["base_commit", "commit", "base_sha", "sha"],
}
ARCHIVE_SUFFIXES = (".tar.gz", ".tgz", ".tar", ".zip")


def _read_rows(p: Path) -> List[dict]:
    txt = p.read_text(encoding="utf-8")
    if p.suffix == ".json":
        d = json.loads(txt)
        return d if isinstance(d, list) else d.get("tasks", d.get("instances", []))
    return [json.loads(l) for l in txt.splitlines() if l.strip()]


def _pick(row: dict, key: str, field_map: Dict[str, str]):
    if key in field_map:
        return row.get(field_map[key])
    for c in FIELD_CANDIDATES[key]:
        if row.get(c) not in (None, ""):
            return row[c]
    return None


def _norm(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", "", s.lower())


def _has_py(d: Path, limit: int = 4000) -> bool:
    n = 0
    for _, _, fns in os.walk(d):
        if any(f.endswith(".py") for f in fns):
            return True
        n += 1
        if n > limit:
            break
    return False


def _git_dir(p: Path) -> Optional[Path]:
    if (p / ".git").exists():
        return p / ".git" if (p / ".git").is_dir() else None   # worktree-file .git not supported
    if (p / "HEAD").is_file() and (p / "objects").is_dir():
        return p                                                # bare repo / mirror
    return None


def _git_archive(cand: Path, commit: str, out: Path) -> bool:
    gd = _git_dir(cand)
    if gd is None or not commit:
        return False
    out.mkdir(parents=True, exist_ok=True)
    tmp = out.parent / f".{out.name}.tar"
    r = subprocess.run(["git", "-c", "safe.directory=*", "--git-dir", str(gd), "archive", "--format=tar",
                        "-o", str(tmp), commit], capture_output=True)
    if r.returncode != 0:
        shutil.rmtree(out, ignore_errors=True)
        return False
    with tarfile.open(tmp) as tf:
        _safe_extract(tf, out)
    tmp.unlink(missing_ok=True)
    return _has_py(out)


def _safe_extract(tf: tarfile.TarFile, out: Path) -> None:
    try:
        tf.extractall(out, filter="data")        # Python >= 3.12 (and backports): blocks path traversal
    except TypeError:
        tf.extractall(out)


def _unpack(p: Path, work: Path, tag: str) -> Optional[Path]:
    out = work / tag
    if not out.exists():
        out.mkdir(parents=True)
        try:
            if p.name.endswith(".zip"):
                zipfile.ZipFile(p).extractall(out)
            else:
                with tarfile.open(p) as tf:
                    _safe_extract(tf, out)
        except Exception:
            shutil.rmtree(out, ignore_errors=True)
            return None
    return _descend(out)


def _descend(d: Path) -> Path:
    """Archives often wrap everything in one top-level folder."""
    subs = [x for x in d.iterdir() if not x.name.startswith(".")]
    if len(subs) == 1 and subs[0].is_dir():
        return subs[0]
    return d


def _repo_keys(repo_id: Optional[str], iid: str) -> List[str]:
    keys: List[str] = []
    if repo_id:
        owner, _, name = repo_id.rpartition("/")
        keys += [repo_id, repo_id.replace("/", "__"), repo_id.replace("/", "_"), repo_id.replace("/", "-"), name]
        if owner:
            keys.append(owner)
    if "_" in iid:
        keys.append(iid.rsplit("_", 1)[0])
    return list(dict.fromkeys(k for k in keys if k))


def _scan_index(snap: Path, maxdepth: int = 3, cap: int = 5000) -> List[Path]:
    out, base = [], len(snap.parts)
    for dp, dns, fns in os.walk(snap):
        depth = len(Path(dp).parts) - base
        dns[:] = [d for d in dns if d != ".git"] if depth < maxdepth else []
        for n in dns + [f for f in fns if f.endswith(ARCHIVE_SUFFIXES)]:
            out.append(Path(dp) / n)
        if len(out) > cap:
            break
    return out


def resolve_snapshot(snap: Optional[Path], tasks_dir: Path, work: Path, iid: str, repo_path: Optional[str],
                     repo_id: Optional[str], commit: Optional[str], _index_cache: dict) -> Tuple[Optional[Path], str]:
    # 0. explicit path column
    if repo_path:
        rp = Path(str(repo_path))
        for c in [rp, tasks_dir / rp] + ([snap / rp] if snap else []):
            if c.is_dir():
                return _descend(c) if not _has_py(c) else c, "per-instance"
            if c.is_file() and c.name.endswith(ARCHIVE_SUFFIXES):
                d = _unpack(c, work, iid)
                if d:
                    return d, "per-instance"
    roots = ([snap] if snap else []) + [tasks_dir / "snapshots"]
    keys = _repo_keys(repo_id, iid)
    for root in roots:
        if not root.exists():
            continue
        # 1. per-instance
        per = [root / iid, root / iid / "repo"] + [root / f"{iid}{s}" for s in ARCHIVE_SUFFIXES] + \
              [root / k / iid for k in keys]
        for c in per:
            if c.is_dir() and _has_py(c):
                return c, "per-instance"
            if c.is_file() and c.name.endswith(ARCHIVE_SUFFIXES):
                d = _unpack(c, work, iid)
                if d and _has_py(d):
                    return d, "per-instance"
        # 2. repo-keyed: git@commit, per-commit folder, plain checkout
        for k in keys:
            for c in (root / k, root / k / "repo", root / k.replace("__", "/")):
                if not c.is_dir():
                    continue
                if commit and _git_dir(c):
                    out = work / f"{iid}_git"
                    if out.exists() and _has_py(out) or _git_archive(c, commit, out):
                        return out, "git@base_commit"
                if commit:
                    for sub in (commit, commit[:12], commit[:7]):
                        if (c / sub).is_dir() and _has_py(c / sub):
                            return c / sub, "per-commit"
                if _has_py(c):
                    return c, "per-repo (commit NOT verified)"
        # 3. fuzzy name match
        idx = _index_cache.setdefault(str(root), _scan_index(root))
        ni = _norm(iid)
        for p in idx:
            nm = _norm(p.name.split(".")[0] if p.is_file() else p.name)
            if nm == ni or (len(ni) >= 6 and ni in nm):
                if p.is_dir() and _has_py(p):
                    return p, "fuzzy"
                if p.is_file():
                    d = _unpack(p, work, iid)
                    if d and _has_py(d):
                        return d, "fuzzy"
    return None, "unresolved"


def describe_layout(snap: Optional[str | Path], max_entries: int = 40, depth: int = 2) -> str:
    if not snap or not Path(snap).exists():
        return f"(snapshots path does not exist: {snap})"
    snap, lines, base = Path(snap), [], len(Path(snap).parts)
    for dp, dns, fns in os.walk(snap):
        d = len(Path(dp).parts) - base
        dns[:] = sorted(x for x in dns if x != ".git") if d < depth else []
        for n in sorted(dns) + sorted(fns)[:5]:
            lines.append("  " * (d + 1) + n + ("/" if n in dns else ""))
        if len(lines) >= max_entries:
            lines.append("  ... (truncated)")
            break
    return f"{snap}\n" + "\n".join(lines[:max_entries + 1])


def load_ids(path: Optional[str]) -> Optional[List[str]]:
    if not path or not Path(path).exists():
        return None
    return [l.strip() for l in Path(path).read_text().splitlines() if l.strip() and not l.startswith("#")]


def load_tasks(tasks_path: str | Path, snapshots_root: Optional[str | Path] = None,
               ids: Optional[List[str]] = None, limit: Optional[int] = None,
               field_map: Optional[Dict[str, str]] = None, work_dir: str | Path = "/tmp/ggil_snapshots"):
    tasks_path, work = Path(tasks_path), Path(work_dir)
    work.mkdir(parents=True, exist_ok=True)
    field_map = field_map or {}
    snap = Path(snapshots_root) if snapshots_root else None
    rows = _read_rows(tasks_path)
    if ids:
        order = {i: n for n, i in enumerate(ids)}
        rows = sorted((r for r in rows if str(_pick(r, "instance_id", field_map)) in order),
                      key=lambda r: order[str(_pick(r, "instance_id", field_map))])
    tasks, cache = [], {}
    report = {"rows": len(rows), "no_issue": 0, "no_patch": 0, "no_repo": 0, "kinds": Counter(),
              "columns": sorted(rows[0]) if rows else []}
    for r in rows:
        iid = str(_pick(r, "instance_id", field_map))
        issue, patch = _pick(r, "issue", field_map), _pick(r, "patch", field_map)
        repo_path, repo_id, commit = _pick(r, "repo", field_map), _pick(r, "repo_id", field_map), _pick(r, "commit", field_map)
        # a mapped 'repo' like "owner/name" is an id, not a path: still tried as a path first, harmlessly
        root, kind = resolve_snapshot(snap, tasks_path.parent, work, iid, repo_path, repo_id or repo_path, commit, cache)
        report["no_issue"] += issue is None
        report["no_patch"] += patch is None
        report["no_repo"] += root is None
        report["kinds"][kind] += 1
        tasks.append({"instance_id": iid, "issue": issue or "", "patch": patch, "repo_root": root,
                      "snapshot_kind": kind, "base_commit": commit, "repo_id": repo_id, "raw": r})
        if limit and len(tasks) >= limit:
            break
    report["kinds"] = dict(report["kinds"])
    if report["no_repo"]:
        t = next((t for t in tasks if t["repo_root"] is None), None)
        report["unresolved_example"] = {k: t[k] for k in ("instance_id", "repo_id", "base_commit")} if t else None
        report["layout"] = describe_layout(snap)
    return tasks, report
''')
sys.path.insert(0, str(_HOT))
import importlib, ggil_data_v2
importlib.reload(ggil_data_v2)
from ggil_data_v2 import load_tasks, load_ids
print('task loader: updated version active')

task loader: updated version active


In [7]:
tp = CFG["TASKS"]
if not tp:
    cands = walk_find(lambda p: p.is_file() and p.suffix in (".jsonl", ".json") and "task" in p.name.lower())
    assert cands, "No tasks file found: add your task data or set CFG['TASKS']."
    if len(cands) > 1:
        print("Several task files found, using the first. Set CFG['TASKS'] to choose:\n  " + "\n  ".join(map(str, cands)))
    tp = str(cands[0])
snap = CFG["SNAPSHOTS"]
if not snap:
    s = walk_find(lambda p: p.is_dir() and p.name in ("snapshots", "repos"))
    snap = str(s[0]) if s else None
ids = load_ids(CFG["ID_FILE"])
tasks, rep = load_tasks(tp, snap, ids=ids, limit=CFG["LIMIT"], field_map=CFG["FIELD_MAP"])
print("tasks file :", tp, "\nsnapshots  :", snap, "\ncolumns    :", rep["columns"])
print({k: v for k, v in rep.items() if k not in ("columns", "layout", "unresolved_example")})

if rep["no_repo"]:
    print("\n--- Some tasks have no snapshot. Example of an unresolved task:", rep.get("unresolved_example"))
    print("--- What the snapshots folder looks like (use this to tell me / set CFG):")
    print(rep.get("layout"))

assert tasks, "No tasks loaded."
assert rep["no_issue"] < len(tasks), "No issue text detected - set CFG['FIELD_MAP']['issue']."
usable = [t for t in tasks if t["repo_root"] is not None]
assert usable, ("No repo snapshots resolved. Read the layout printed above: the loader supports per-task folders/archives, "
                "git clones or mirrors (checked out at base_commit), and per-commit folders. Tell me the layout if it is different.")

unverified = [t for t in usable if "NOT verified" in t["snapshot_kind"]]
if unverified and not CFG.get("ALLOW_UNVERIFIED", False):
    print(f"WARNING: {len(unverified)} tasks only have a per-repo checkout whose commit cannot be verified against base_commit; "
          "gold line numbers may not match. They are EXCLUDED. Set CFG['ALLOW_UNVERIFIED']=True to keep them anyway.")
    usable = [t for t in usable if t not in unverified]
    assert usable, "All snapshots are unverified per-repo checkouts - see warning above."
elif unverified:
    print(f"NOTE: keeping {len(unverified)} unverified per-repo snapshots (ALLOW_UNVERIFIED=True) - treat results with caution.")
if len(usable) < len(tasks):
    print(f"{len(tasks) - len(usable)} of {len(tasks)} tasks dropped (no usable snapshot).")
tasks = usable
HAS_GOLD = any(t["patch"] for t in tasks)
print(f"{len(tasks)} tasks | gold patches available: {HAS_GOLD} | snapshot kinds: {rep['kinds']}")

tasks file : /kaggle/input/competitions/gemma-4-developer-agent/tasks.jsonl 
snapshots  : /kaggle/input/competitions/gemma-4-developer-agent/snapshots 
columns    : ['base_commit', 'created_at', 'hints_text', 'instance_id', 'patch', 'problem_statement', 'repo', 'test_patch']
{'rows': 129, 'no_issue': 0, 'no_patch': 0, 'no_repo': 0, 'kinds': {'per-instance': 129}}
129 tasks | gold patches available: True | snapshot kinds: {'per-instance': 129}


## 3. Phase 1 — build graphs, extract gold, check data quality
Graphs must be built on the **pre-fix** snapshot (gold line numbers refer to the old side of the patch). The check below flags snapshots where patched `.py` files are missing, which usually means the wrong commit was checked out.

In [8]:
import pandas as pd

G, t0 = {}, time.time()
for t in tasks:
    g = build_graph(t["repo_root"])
    gold = gold_from_patch(g, t["patch"]) if t["patch"] else None
    mismatch = 0
    if t["patch"]:
        for fp in parse_patch(t["patch"]):
            p = fp.old_path
            if p and not fp.is_new and p.endswith(".py") and p not in g.nodes and "test" not in p.lower():
                mismatch += 1
    G[t["instance_id"]] = dict(graph=g, gold=gold, tools=GraphTools(g), mismatch=mismatch)
print(f"built {len(G)} graphs in {time.time() - t0:.1f}s")

rows = []
for t in tasks:
    d, g = G[t["instance_id"]], G[t["instance_id"]]["graph"]
    st = g.stats()
    rows.append({"instance_id": t["instance_id"], "files": st["nodes"].get("file", 0),
                 "symbols": sum(v for k, v in st["nodes"].items() if k != "file"),
                 "calls_exact": st["edges"].get("calls:exact", 0), "calls_heur": st["edges"].get("calls:heuristic", 0),
                 "gold_files": len(d["gold"].files) if d["gold"] else None,
                 "gold_units": len(d["gold"].units) if d["gold"] else None,
                 "snapshot_mismatch": d["mismatch"]})
stats_df = pd.DataFrame(rows)
display(stats_df.describe().T[["mean", "min", "max"]].round(1))
if HAS_GOLD:
    n_u = int((stats_df.gold_units.fillna(0) > 0).sum()); n_f = int((stats_df.gold_files.fillna(0) > 0).sum())
    print(f"tasks with gold files: {n_f}/{len(tasks)} | with gold function-level units: {n_u}/{len(tasks)} "
          f"(function-level metrics are computed only on these {n_u})")
    bad = int((stats_df.snapshot_mismatch > 0).sum())
    if bad:
        print(f"WARNING: {bad} tasks have patched .py files missing from the snapshot - check that snapshots are the PRE-fix commit.")

built 129 graphs in 262.6s


,mean,min,max
files,690.6,35.0,1279.0
symbols,3479.7,727.0,5845.0
calls_exact,2997.9,816.0,3635.0
calls_heur,1571.4,299.0,2185.0
gold_files,1.8,0.0,20.0
gold_units,4.1,0.0,139.0
snapshot_mismatch,0.4,0.0,23.0


tasks with gold files: 128/129 | with gold function-level units: 121/129 (function-level metrics are computed only on these 121)


## 4. Phase 1 — no-LLM baselines and graph ablation ladder

In [9]:
def score_rows(preds):
    """preds: {iid: (pred_files, pred_units)} -> list of per-instance metric dicts (gold-bearing tasks only)."""
    out = []
    for t in tasks:
        d = G[t["instance_id"]]
        if d["gold"] is None or not (d["gold"].files or d["gold"].units):
            continue
        pf, pu = preds.get(t["instance_id"], ([], []))
        m = score_instance(d["graph"], d["gold"], pf or None, pu)
        m["instance_id"] = t["instance_id"]
        out.append(m)
    return out

RESULTS = {}     # system -> per-instance rows
EXTRA = {}       # system -> {"answered": .., "tool_calls": ..}
if HAS_GOLD:
    ladder = {"BM25": dict(propagate=0.0, use_signals=False),
              "BM25+signals": dict(propagate=0.0, use_signals=True),
              "GraphRank": dict(propagate=0.3, use_signals=True)}
    for name, kw in ladder.items():
        preds = {}
        for t in tasks:
            r = G[t["instance_id"]]["tools"].rank(t["issue"], **kw)
            preds[t["instance_id"]] = ([f for f, _ in r.files[:10]], [u for u, _ in r.units[:10]])
        RESULTS[name] = score_rows(preds)
    print("done:", list(RESULTS))
else:
    print("No gold patches in this task file: scoring skipped (rankings can still be produced for submission).")

done: ['BM25', 'BM25+signals', 'GraphRank']


## 4A. Primary/clean cohorts and held-out split

A deterministic 80/20 task split is used only for selecting the graph propagation weight. The selected value is then frozen before held-out evaluation. The original 0.30 setting remains a non-tuned reference.


In [10]:
import hashlib

def split_label(iid):
    x=int(hashlib.sha256((iid+"::"+str(CFG["RANDOM_SEED"])).encode()).hexdigest()[:8],16)/0xFFFFFFFF*100
    return "dev" if x<20 else "test"

cohort_df=stats_df.copy()
cohort_df["cohort"]=cohort_df["instance_id"].map(split_label)
cohort_df["clean_snapshot"]=cohort_df["snapshot_mismatch"].eq(0)
display(cohort_df.groupby(["cohort","clean_snapshot"]).size().rename("n").reset_index())
print("Resolvable:",len(cohort_df)," Clean:",int(cohort_df.clean_snapshot.sum()))


,cohort,clean_snapshot,n
0,dev,False,1
1,dev,True,24
2,test,False,10
3,test,True,94


Resolvable: 129  Clean: 118


## 4B. Propagation-weight sensitivity

The original notebook fixed the graph weight at 0.30. We now evaluate a prespecified grid on development tasks and freeze the best value by function-level Acc@1, with MRR as the tie-breaker.


In [11]:
def score_rows_for_ids(preds,ids):
    ids=set(ids); out=[]
    for t in tasks:
        iid=t["instance_id"]
        if iid not in ids: continue
        d=G[iid]
        if d["gold"] is None or not (d["gold"].files or d["gold"].units): continue
        pf,pu=preds.get(iid,([],[]))
        m=score_instance(d["graph"],d["gold"],pf or None,pu); m["instance_id"]=iid; out.append(m)
    return out

def run_ranker(propagate,use_signals=True,ids=None):
    ids=set(ids) if ids is not None else {t["instance_id"] for t in tasks}; preds={}
    for t in tasks:
        iid=t["instance_id"]
        if iid not in ids: continue
        r=G[iid]["tools"].rank(t["issue"],propagate=propagate,use_signals=use_signals)
        preds[iid]=([f for f,_ in r.files[:10]],[u for u,_ in r.units[:10]])
    return score_rows_for_ids(preds,ids)

split_map=dict(zip(cohort_df.instance_id,cohort_df.cohort))
dev_ids=[i for i,v in split_map.items() if v=="dev"]; test_ids=[i for i,v in split_map.items() if v=="test"]
alpha_rows=[]
for alpha in CFG["ALPHA_GRID"]:
    rr=run_ranker(alpha,True,dev_ids); a=aggregate(rr,n_boot=CFG["BOOTSTRAP"])
    if "unit/acc@1" in a:
        alpha_rows.append({"alpha":alpha,"n":a["unit/acc@1"]["n"],"dev_acc1":a["unit/acc@1"]["mean"],"dev_mrr":a.get("unit/mrr",{}).get("mean")})
alpha_df=pd.DataFrame(alpha_rows); display(alpha_df)
selected_alpha=float(alpha_df.sort_values(["dev_acc1","dev_mrr"],ascending=False).iloc[0]["alpha"]) if len(alpha_df) else CFG["PROPAGATION_WEIGHT"]
print("Frozen alpha:",selected_alpha)
frozen_rows=run_ranker(selected_alpha,True,test_ids) if test_ids else []
frozen_agg=aggregate(frozen_rows,n_boot=CFG["BOOTSTRAP"]) if frozen_rows else {}
print("Held-out:",frozen_agg.get("unit/acc@1"))
alpha_df.to_csv(CFG["OUT"]/"propagation_weight_sensitivity.csv",index=False)


,alpha,n,dev_acc1,dev_mrr
0,0.0,24,0.208333,0.259491
1,0.1,24,0.208333,0.267179
2,0.2,24,0.208333,0.272388
3,0.3,24,0.250000,0.307060
4,0.4,24,0.250000,0.310417
5,0.5,24,0.250000,0.311111
6,0.7,24,0.250000,0.288194
7,0.9,24,0.166667,0.231647


Frozen alpha: 0.5
Held-out: {'mean': 0.23711340206185566, 'n': 97, 'ci_lo': 0.15463917525773196, 'ci_hi': 0.32989690721649484}


## 4C. Graph relation ablation

We attempt calls-only, imports-only, inheritance/definition-only, and all-edge conditions. The experiment is capability-adaptive so an unsupported graph representation is reported as unavailable rather than approximated.


In [12]:
print("Graph type:",type(next(iter(G.values()))["graph"]))
print("GraphTools.rank:",inspect.signature(GraphTools.rank))
EDGE_KEYS=("type","kind","relation","edge_type")
def edge_type(d):
    if isinstance(d,dict):
        for k in EDGE_KEYS:
            if k in d and d[k] is not None: return str(d[k]).lower()
    return ""
def filtered_graph(g,allowed):
    try:
        h=copy.deepcopy(g)
        if not (hasattr(h,"edges") and hasattr(h,"remove_edge")): return None
        allowed={x.lower() for x in allowed}
        edges=list(h.edges(keys=True,data=True))
        for u,v,k,d in edges:
            typ=edge_type(d)
            if typ and not any(a in typ for a in allowed):
                try: h.remove_edge(u,v,k)
                except TypeError: h.remove_edge(u,v)
        return h
    except Exception as e:
        print("filter failed",repr(e)); return None

relation_sets={"calls_only":["call"],"imports_only":["import"],"inheritance_only":["inherit","defined_in","definition"],"all":[]}
relation_results=[]
for rel,allowed in relation_sets.items():
    preds={}; ran=0
    for t in tasks:
        iid=t["instance_id"]; g0=G[iid]["graph"]
        h=g0 if rel=="all" else filtered_graph(g0,allowed)
        if h is None: continue
        try:
            r=GraphTools(h).rank(t["issue"],propagate=selected_alpha,use_signals=True)
            preds[iid]=([f for f,_ in r.files[:10]],[u for u,_ in r.units[:10]]); ran+=1
        except Exception: pass
    if ran:
        a=aggregate(score_rows(preds),n_boot=CFG["BOOTSTRAP"])
        relation_results.append({"graph":rel,"tasks":ran,"unit_acc1":a.get("unit/acc@1",{}).get("mean"),"unit_mrr":a.get("unit/mrr",{}).get("mean")})
    else: relation_results.append({"graph":rel,"tasks":0,"unit_acc1":None,"unit_mrr":None})
relation_df=pd.DataFrame(relation_results); display(relation_df); relation_df.to_csv(CFG["OUT"]/"graph_relation_ablation.csv",index=False)


Graph type: <class 'ggil.graph.RepoGraph'>
GraphTools.rank: (self, issue_text: 'str', propagate: 'float' = 0.3, seed_k: 'int' = 30, include_tests: 'bool' = False, use_signals: 'bool' = True) -> 'Ranking'


,graph,tasks,unit_acc1,unit_mrr
0,calls_only,0,NaN,NaN
1,imports_only,0,NaN,NaN
2,inheritance_only,0,NaN,NaN
3,all,129,0.239669,0.316614


## 4D. Snapshot sensitivity, graph noise, and repository scale


In [13]:
def subset_rows(rows,ids):
    ids=set(ids); return [r for r in rows if r.get("instance_id") in ids]
clean_ids=set(cohort_df.loc[cohort_df.clean_snapshot,"instance_id"])
robust=[]
for system,rows in RESULTS.items():
    for label,ids in [("all",set(cohort_df.instance_id)),("clean_snapshot",clean_ids)]:
        rr=subset_rows(rows,ids)
        if not rr: continue
        a=aggregate(rr,n_boot=CFG["BOOTSTRAP"])
        robust.append({"system":system,"cohort":label,"n":a.get("unit/acc@1",{}).get("n",0),"unit_acc1":a.get("unit/acc@1",{}).get("mean"),"unit_mrr":a.get("unit/mrr",{}).get("mean")})
robust_df=pd.DataFrame(robust); display(robust_df); robust_df.to_csv(CFG["OUT"]/"snapshot_sensitivity.csv",index=False)

stats_df["heuristic_call_fraction"]=stats_df["calls_heur"]/(stats_df["calls_exact"]+stats_df["calls_heur"]).replace(0,np.nan)
stats_df["repo_size_bin"]=pd.qcut(stats_df["files"],q=4,duplicates="drop",labels=["Q1-small","Q2","Q3","Q4-large"])
scale=[]
for b,grp in stats_df.groupby("repo_size_bin",observed=True):
    ids=set(grp.instance_id)
    for system in ["BM25+signals","GraphRank"]:
        rr=subset_rows(RESULTS.get(system,[]),ids)
        if not rr: continue
        a=aggregate(rr,n_boot=CFG["BOOTSTRAP"])
        scale.append({"repo_size_bin":str(b),"system":system,"n":a.get("unit/acc@1",{}).get("n",0),"unit_acc1":a.get("unit/acc@1",{}).get("mean")})
scale_df=pd.DataFrame(scale); display(scale_df); scale_df.to_csv(CFG["OUT"]/"repository_scale_analysis.csv",index=False)


,system,cohort,n,unit_acc1,unit_mrr
0,BM25,all,121,0.132231,0.208222
1,BM25,clean_snapshot,111,0.144144,0.216470
2,BM25+signals,all,121,0.223140,0.288584
3,BM25+signals,clean_snapshot,111,0.243243,0.304072
4,GraphRank,all,121,0.247934,0.324823
5,GraphRank,clean_snapshot,111,0.270270,0.336883


,repo_size_bin,system,n,unit_acc1
0,Q1-small,BM25+signals,43,0.325581
1,Q1-small,GraphRank,43,0.348837
2,Q2,BM25+signals,17,0.235294
3,Q2,GraphRank,17,0.235294
4,Q3,BM25+signals,31,0.161290
5,Q3,GraphRank,31,0.193548
6,Q4-large,BM25+signals,30,0.133333
7,Q4-large,GraphRank,30,0.166667


## 4E. Task-level graph gains


In [14]:
def metric_map(rows, metric):
    return {r["instance_id"]: r.get(metric) for r in rows}

if "GraphRank" in RESULTS and "BM25+signals" in RESULTS:
    gm = metric_map(RESULTS["GraphRank"], "unit/acc@1")
    sm = metric_map(RESULTS["BM25+signals"], "unit/acc@1")
    
    # Filter out entries where either score is None
    valid_ids = [i for i in sorted(set(gm) & set(sm)) if gm[i] is not None and sm[i] is not None]
    
    tg = pd.DataFrame([{
        "instance_id": i, 
        "graph_minus_signals": float(gm[i]) - float(sm[i]) 
    } for i in valid_ids])
    
    tg = tg.merge(stats_df[["instance_id", "files", "symbols", "calls_exact", "calls_heur", "heuristic_call_fraction"]], on="instance_id", how="left")
    tg["graph_helps"] = tg.graph_minus_signals > 0
    display(tg.head(20))
    print("wins/ties/losses:", int((tg.graph_minus_signals > 0).sum()), int((tg.graph_minus_signals == 0).sum()), int((tg.graph_minus_signals < 0).sum()))
    tg.to_csv(CFG["OUT"] / "task_level_graph_gains.csv", index=False)

,instance_id,graph_minus_signals,files,symbols,calls_exact,calls_heur,heuristic_call_fraction,graph_helps
0,fastapi_11194,0.0,1135,4371,2962,879,0.228847,False
1,fastapi_11355,0.0,1202,5020,3309,1197,0.265646,False
2,fastapi_12942,0.0,1142,4439,2998,884,0.227718,False
3,fastapi_13207,0.0,1198,4913,3273,1194,0.267293,False
4,fastapi_13537,0.0,1194,4879,3261,1176,0.265044,False
5,fastapi_13713,0.0,1133,4350,2946,879,0.229804,False
6,fastapi_13786,0.0,1186,4830,3236,1175,0.266380,False
7,fastapi_13920,0.0,1255,5497,2994,1729,0.366081,False
8,fastapi_14077,0.0,1133,4350,2946,879,0.229804,False
9,fastapi_14099,0.0,1136,4379,2964,885,0.229930,False


wins/ties/losses: 4 116 1


In [15]:
from pathlib import Path

W = Path("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse")

print("Exists:", W.exists())

for f in W.glob("*.whl"):
    print(f.name)

Exists: True
safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
mistral_common-1.11.7-py3-none-any.whl
partial_json_parser-0.2.1.1.post7-py3-none-any.whl
compressed_tensors-0.15.0.1-py3-none-any.whl
setproctitle-1.3.7-cp312-cp312-manylinux1_x86_64.manylinux_2_28_x86_64.manylinux_2_5_x86_64.whl
adk_submission-0.2.12-py3-none-any.whl
model_hosting_container_standards-0.1.16-py3-none-any.whl
interegular-0.3.3-py37-none-any.whl
outlines_core-0.2.11-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
msgspec-0.21.1-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl
pycountry-26.2.16-py3-none-any.whl
adk_eval_core-0.1.0-py3-none-any.whl
quack_kernels-0.6.4-py3-none-any.whl
pydantic_extra_types-2.11.2-py3-none-any.whl
transformers-5.13.1-py3-none-any.whl
apache_tvm_ffi-0.1.13.post3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl
google_adk-1.36.1-py3-none-any.whl
google_genai-2.11.0-py3-none-any.whl
gguf-0.19.0-py3-none

## 5. Phase 2 — controlled LLM agents (A / S / D)

All variants share the same model, decoding settings, repository snapshot, context length and maximum step budget. Only graph access changes:

- **A:** file tools only.
- **S:** A + static graph-derived hints.
- **D:** A + interactive graph tools.

The notebook records localization accuracy, answer rate, tool calls and runtime. Failed/unparseable answers remain misses; nothing is back-filled from a ranker.


In [16]:
LLM_READY,llm,sampling,why_skipped=False,None,None,None
try:
    import torch; n_gpu=torch.cuda.device_count()
except Exception: n_gpu=0
mp=CFG["MODEL_PATH"]
if not mp:
    cands=[p.parent for p in walk_find(lambda p:p.is_file() and p.name=="config.json",maxdepth=7)]
    cands=[c for c in cands if any(c.glob("*.safetensors")) or any(c.glob("*.bin"))]
    if len(cands)==1: mp=str(cands[0])
    elif cands:
        print("Set CFG['MODEL_PATH'] to one of:")
        print("\n  ".join(map(str,cands)))
if n_gpu==0: why_skipped="no GPU visible"
elif not mp: why_skipped="no model found"
else:
    try:
        os.environ.update(HF_HUB_OFFLINE="1",TRANSFORMERS_OFFLINE="1")
        wh=Path("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse")
        if wh.exists(): subprocess.run([sys.executable,"-m","pip","install","--no-index","--find-links",str(wh),"msgspec"],check=False)
        from vllm import LLM,SamplingParams
        tp=min(CFG["TENSOR_PARALLEL_SIZE"],n_gpu)
        llm=LLM(model=mp,tensor_parallel_size=tp,max_model_len=CFG["MAX_MODEL_LEN"],gpu_memory_utilization=CFG["GPU_MEM_UTIL"])
        sampling=SamplingParams(temperature=CFG["TEMPERATURE"],max_tokens=CFG["MAX_NEW_TOKENS"],seed=CFG["RANDOM_SEED"])
        LLM_READY=True
    except Exception as e: why_skipped=f"{type(e).__name__}: {str(e)[:500]}"
print(f"GPUs={n_gpu} | TP={min(CFG['TENSOR_PARALLEL_SIZE'],n_gpu) if n_gpu else 0} | model={mp} | LLM_READY={LLM_READY}")
if why_skipped: print("Phase 2 unavailable:",why_skipped)


ERROR: Could not find a version that satisfies the requirement msgspec (from versions: none)
ERROR: No matching distribution found for msgspec


Looking in links: /kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse
GPUs=4 | TP=4 | model=/kaggle/input/models/thomasplocher/gemma-4-31b-w4a16-ct-mirror/transformers/gemma-4-31b-it-qat-w4a16-ct/1 | LLM_READY=False
Phase 2 unavailable: ModuleNotFoundError: No module named 'msgspec'


In [17]:
RAW_PREDS = {}
if LLM_READY:
    deadline = time.time() + CFG["MAX_MINUTES"] * 60
    ctxs = [TaskCtx(t["instance_id"], t["issue"], Path(t["repo_root"]), G[t["instance_id"]]["graph"], G[t["instance_id"]]["tools"]) for t in tasks]
    for v in CFG["VARIANTS"]:
        t1 = time.time()
        res = run_variant(llm, sampling, ctxs, v, max_steps=CFG["MAX_STEPS"], deadline=deadline)
        RAW_PREDS[v] = res
        with open(CFG["OUT"] / f"agent_{v}_preds.jsonl", "w") as f:
            for r in res:
                f.write(json.dumps(r) + "\n")
        print(f"variant {v}: {time.time() - t1:.0f}s | answered {sum(r['answered'] for r in res)}/{len(res)} | "
              f"errors { {e: sum(r['error'] == e for r in res) for e in set(r['error'] for r in res if r['error'])} }")
        if HAS_GOLD:
            RESULTS[f"LLM-{v}"] = score_rows({r["instance_id"]: (r["pred_files"], r["pred_units"]) for r in res})
            EXTRA[f"LLM-{v}"] = {"answered": sum(r["answered"] for r in res) / len(res),
                                 "tool_calls": sum(r["tool_calls"] for r in res) / len(res)}
else:
    print("Phase 2 skipped:", why_skipped)

Phase 2 skipped: ModuleNotFoundError: No module named 'msgspec'


## 5A. Optional agent-budget sensitivity

Set `RUN_BUDGET_SWEEP=True` only after the primary A/S/D run succeeds. This reruns the same variants at fixed step budgets and records localization accuracy, answer rate, tool calls, and runtime. It directly tests whether graph access gives more useful search under constrained interaction budgets.


In [18]:
RUN_BUDGET_SWEEP=False
BUDGETS=[4,8,12,16]
budget_rows=[]
if RUN_BUDGET_SWEEP and LLM_READY:
    ctxs=[TaskCtx(t["instance_id"],t["issue"],Path(t["repo_root"]),G[t["instance_id"]]["graph"],G[t["instance_id"]]["tools"]) for t in tasks]
    for budget in BUDGETS:
        deadline=time.time()+CFG["MAX_MINUTES"]*60
        for v in CFG["VARIANTS"]:
            t0=time.time(); res=run_variant(llm,sampling,ctxs,v,max_steps=budget,deadline=deadline); elapsed=time.time()-t0
            rows=score_rows({r["instance_id"]:(r["pred_files"],r["pred_units"]) for r in res})
            a=aggregate(rows,n_boot=CFG["BOOTSTRAP"])
            budget_rows.append({"variant":v,"max_steps":budget,"unit_acc1":a.get("unit/acc@1",{}).get("mean"),"unit_mrr":a.get("unit/mrr",{}).get("mean"),"answer_rate":sum(r["answered"] for r in res)/len(res),"mean_tool_calls":sum(r["tool_calls"] for r in res)/len(res),"runtime_sec":elapsed})
    budget_df=pd.DataFrame(budget_rows); display(budget_df); budget_df.to_csv(CFG["OUT"]/"agent_budget_sensitivity.csv",index=False)
elif RUN_BUDGET_SWEEP:
    print("Budget sweep requested but Phase 2 is unavailable:",why_skipped)
else:
    print("Budget sweep disabled. Set RUN_BUDGET_SWEEP=True after a successful primary A/S/D run.")


Budget sweep disabled. Set RUN_BUDGET_SWEEP=True after a successful primary A/S/D run.


## 6. Results

In [19]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SHOW = ["file/acc@1", "file/acc@5", "unit/acc@1", "unit/acc@3", "unit/acc@5", "unit/acc@10", "unit/mrr", "unit/recall@5"]
AGG, table = {}, []
for name, rows in RESULTS.items():
    if not rows:
        continue
    a = aggregate(rows, n_boot=CFG["BOOTSTRAP"])
    AGG[name] = a
    r = {"system": name, "n_file": a.get("file/acc@1", {}).get("n", 0), "n_unit": a.get("unit/acc@1", {}).get("n", 0)}
    for k in SHOW:
        if k in a:
            r[k] = f"{a[k]['mean']:.3f} [{a[k]['ci_lo']:.2f},{a[k]['ci_hi']:.2f}]"
    r.update({k: round(v, 2) for k, v in EXTRA.get(name, {}).items()})
    table.append(r)
if table:
    res_df = pd.DataFrame(table).set_index("system")
    display(res_df)
    res_df.to_csv(CFG["OUT"] / "results_table.csv")
else:
    print("No scored results (no gold patches or no systems ran).")

,n_file,n_unit,file/acc@1,file/acc@5,unit/acc@1,unit/acc@3,unit/acc@5,unit/acc@10,unit/mrr,unit/recall@5
system,,,,,,,,,,
BM25,128,121,"0.305 [0.23,0.38]","0.664 [0.59,0.74]","0.132 [0.07,0.20]","0.264 [0.19,0.35]","0.306 [0.22,0.39]","0.413 [0.33,0.50]","0.208 [0.15,0.27]","0.189 [0.13,0.25]"
BM25+signals,128,121,"0.375 [0.30,0.46]","0.688 [0.61,0.77]","0.223 [0.15,0.31]","0.339 [0.26,0.42]","0.372 [0.29,0.46]","0.463 [0.38,0.55]","0.289 [0.22,0.36]","0.260 [0.19,0.33]"
GraphRank,128,121,"0.430 [0.34,0.52]","0.719 [0.64,0.80]","0.248 [0.17,0.32]","0.364 [0.28,0.45]","0.421 [0.34,0.51]","0.512 [0.43,0.60]","0.325 [0.25,0.40]","0.284 [0.21,0.36]"


In [20]:
if len(AGG) >= 2:
    mets = ["unit/acc@1", "unit/acc@3", "unit/acc@5"]
    fig, ax = plt.subplots(figsize=(9, 4))
    w = 0.8 / len(AGG)
    for i, (name, a) in enumerate(AGG.items()):
        xs = [j + i * w for j in range(len(mets))]
        ys = [a[m]["mean"] for m in mets]
        err = [[a[m]["mean"] - a[m]["ci_lo"] for m in mets], [a[m]["ci_hi"] - a[m]["mean"] for m in mets]]
        ax.bar(xs, ys, w, yerr=err, capsize=2, label=name)
    ax.set_xticks([j + 0.4 - w / 2 for j in range(len(mets))]); ax.set_xticklabels(mets)
    ax.set_ylabel("fraction of tasks"); ax.set_ylim(0, 1); ax.legend(fontsize=8, ncol=2)
    ax.set_title("Function-level localization (95% bootstrap CI)")
    fig.tight_layout(); fig.savefig(CFG["OUT"] / "unit_acc.png", dpi=150); plt.show()

In [21]:
# Paired comparisons with Holm correction.
pairs=[("BM25","BM25+signals"),("BM25+signals","GraphRank"),("BM25","GraphRank"),("LLM-A","LLM-S"),("LLM-A","LLM-D"),("LLM-S","LLM-D")]
out=[]
for a,b in pairs:
    if a not in RESULTS or b not in RESULTS: continue
    for m in ["unit/acc@1","unit/acc@5","unit/mrr","file/acc@1"]:
        try:
            r=paired_bootstrap(RESULTS[a],RESULTS[b],m)
            out.append({"comparison":f"{b} - {a}","metric":m,"n":r["n"],"diff":r["mean_diff"],"ci_lo":r["ci_lo"],"ci_hi":r["ci_hi"],"p":r["p_value"]})
        except Exception: pass
cmp_df=pd.DataFrame(out)
if len(cmp_df):
    order=cmp_df["p"].argsort().tolist(); m=len(cmp_df); adj=[None]*m; running=0.0
    for rank,pos in enumerate(order):
        running=max(running,min(1.0,(m-rank)*float(cmp_df.iloc[pos]["p"]))); adj[pos]=running
    cmp_df["p_holm"]=adj; cmp_df["significant_0.05"]=cmp_df.p_holm<0.05
    cmp_df["ci"]=cmp_df.apply(lambda r:f"[{r.ci_lo:.3f},{r.ci_hi:.3f}]",axis=1)
    display(cmp_df[["comparison","metric","n","diff","ci","p","p_holm","significant_0.05"]])
    cmp_df.to_csv(CFG["OUT"]/"paired_comparisons_holm.csv",index=False)
else: print("No paired comparisons available.")
        

,comparison,metric,n,diff,ci,p,p_holm,significant_0.05
0,BM25+signals - BM25,unit/acc@1,121,0.090909,"[0.041,0.149]",0.0000,0.0000,True
1,BM25+signals - BM25,unit/acc@5,121,0.066116,"[0.025,0.116]",0.0000,0.0000,True
2,BM25+signals - BM25,unit/mrr,121,0.080362,"[0.039,0.126]",0.0000,0.0000,True
3,BM25+signals - BM25,file/acc@1,128,0.070312,"[0.023,0.125]",0.0024,0.0120,True
4,GraphRank - BM25+signals,unit/acc@1,121,0.024793,"[-0.008,0.066]",0.2444,0.2444,False
5,GraphRank - BM25+signals,unit/acc@5,121,0.049587,"[0.008,0.099]",0.0288,0.0864,False
6,GraphRank - BM25+signals,unit/mrr,121,0.036239,"[0.011,0.066]",0.0044,0.0176,True
7,GraphRank - BM25+signals,file/acc@1,128,0.054688,"[0.008,0.109]",0.0388,0.0864,False
8,GraphRank - BM25,unit/acc@1,121,0.115702,"[0.050,0.182]",0.0004,0.0024,True
9,GraphRank - BM25,unit/acc@5,121,0.115702,"[0.066,0.174]",0.0000,0.0000,True


## 7. Save everything (measured values only)

In [22]:
def _ver(m):
    try:
        return __import__(m).__version__
    except Exception:
        return None

manifest = dict(
    study_design="Q1-upgraded",
    n_tasks=len(tasks), instance_ids=[t["instance_id"] for t in tasks], has_gold=HAS_GOLD,
    systems=list(RESULTS), phase2_ran=LLM_READY, phase2_skip_reason=why_skipped,
    config={k: (str(v) if isinstance(v, Path) else v) for k, v in CFG.items()},
    model_path=mp if LLM_READY else None, gpus=n_gpu,
    gpu_name=(__import__("torch").cuda.get_device_name(0) if n_gpu else None) if "torch" in sys.modules else None,
    versions=dict(python=platform.python_version(), vllm=_ver("vllm"), torch=_ver("torch")),
    toolkit_sha256=hashlib.sha256(tk.read_bytes()).hexdigest() if tk.is_file() else None,
    timestamp=time.strftime("%Y-%m-%d %H:%M:%S"),
    seed=CFG["RANDOM_SEED"], selected_alpha=globals().get("selected_alpha"),
)
(CFG["OUT"] / "run_manifest.json").write_text(json.dumps(manifest, indent=1))
(CFG["OUT"] / "per_instance.json").write_text(json.dumps(RESULTS))
(CFG["OUT"] / "aggregate.json").write_text(json.dumps(AGG, indent=1))
stats_df.to_csv(CFG["OUT"] / "graph_stats.csv", index=False)
print("saved to", CFG["OUT"], "->", sorted(p.name for p in CFG["OUT"].iterdir()))

if CFG["PACK"]:
    zp = CFG["OUT"].parent / "ggil_bundle.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        for p in Path(pkg_root / "ggil").rglob("*.py"):
            z.write(p, Path("ggil") / p.relative_to(pkg_root / "ggil"))
        for p in CFG["OUT"].iterdir():
            z.write(p, Path("results") / p.name)
    print("bundle:", zp, "(NOTE: this is not an official competition submission; follow the competition's required layout)")

saved to /kaggle/working/ggil_results_q1 -> ['aggregate.json', 'graph_relation_ablation.csv', 'graph_stats.csv', 'paired_comparisons_holm.csv', 'per_instance.json', 'propagation_weight_sensitivity.csv', 'repository_scale_analysis.csv', 'results_table.csv', 'run_manifest.json', 'snapshot_sensitivity.csv', 'task_level_graph_gains.csv', 'unit_acc.png']
